# Extraction Results — Exact Match Evaluation

Micro-averaged Precision, Recall, F1 per category and overall.

**Micro-averaging** pools raw triple counts (matched, predicted, gold) before dividing — the correct approach when entries contain very few triples (1–3) and categories have different sizes.

In [5]:
import json
import pandas as pd
from collections import defaultdict
import re

RESULTS_PATH = "results/results_gemini_gemini-3.1-flash-lite-preview_rdf_shacl_1.jsonl"

# ── Load results ──────────────────────────────────────────────────────────────
records = []
with open(RESULTS_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
df["category"] = df["entry_id"].str.rsplit("_", n=2).str[0]
df["n_gold"] = df["gold_triples"].apply(len)
df["n_pred"] = df["pred_triples"].apply(len)

def add_underscores(text):
    """Replace any non-alphanumeric symbols that separate words with underscores, keep numbers."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', text).strip('_')

def process_triple(triple):
    """Add underscores to subject and object in a triple."""
    return {
        'subject': add_underscores(triple['subject']),
        'relation': triple['relation'],
        'object': add_underscores(triple['object'])
    }

# Apply to all gold triples
df['gold_triples'] = df['gold_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)

# Reapply to predicted triples to use updated function
df['pred_triples'] = df['pred_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)


print(f"Loaded {len(df)} entries across {df['category'].nunique()} categories")
print(f"Gold triples: {df['n_gold'].sum()}  |  Predicted triples: {df['n_pred'].sum()}")
df[["entry_id", "category", "n_gold", "n_pred"]].head(10)

Loaded 990 entries across 57 categories
Gold triples: 1919  |  Predicted triples: 1831


,entry_id,category,n_gold,n_pred
0,1_Airport_dev_1,1_Airport,1,1
1,1_Airport_dev_2,1_Airport,1,1
2,1_Airport_dev_3,1_Airport,1,1
3,1_Airport_dev_4,1_Airport,1,1
4,1_Airport_dev_5,1_Airport,1,1
5,1_Airport_dev_6,1_Airport,1,1
6,1_Airport_dev_7,1_Airport,1,1
7,1_Airport_dev_8,1_Airport,1,1
8,1_Airport_dev_9,1_Airport,1,1
9,1_Airport_dev_10,1_Airport,1,1


# SHACL-style Schema Violation Analysis

For each predicted triple, check whether the predicted **subject type** and **object type** (from `pred_schemas`) satisfy the **domain** and **range** constraints declared in the ontology JSON.

Uses the hierarchy (IsA relations) for transitive subclass matching — e.g. `City` is-a `Settlement` is-a `PopulatedPlace` is-a `Place`.

In [15]:
import json, os, re
from collections import defaultdict
from rdflib import Graph
from rdflib.namespace import OWL, RDF, RDFS

ONTOLOGY_DIR = "OSKGC/ontologies/json"
RDF_ONTOLOGY_DIR = "OSKGC/ontologies/rdf"

# ── Build a global URI-local-name → label map from RDF ontologies ────────────
_uri_to_label: dict = {}
for _fn in os.listdir(RDF_ONTOLOGY_DIR):
    if not _fn.endswith(".ttl"):
        continue
    _g = Graph()
    _g.parse(os.path.join(RDF_ONTOLOGY_DIR, _fn), format="turtle")
    for _cls in _g.subjects(RDF.type, OWL.Class):
        uri_str = str(_cls)
        local = uri_str.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
        for _lbl in _g.objects(_cls, RDFS.label):
            _uri_to_label[local] = str(_lbl)
            _uri_to_label[str(_lbl)] = str(_lbl)
            _uri_to_label[uri_str] = str(_lbl)

for _fn in os.listdir(ONTOLOGY_DIR):
    if not _fn.endswith(".json"):
        continue
    _ont = json.load(open(os.path.join(ONTOLOGY_DIR, _fn), "r", encoding="utf-8"))
    for _e in _ont.get("entity type", []):
        _uri_to_label[_e["id"]] = _e["label"]
        _uri_to_label[_e["label"]] = _e["label"]

print(f"URI→label map: {len(_uri_to_label)} entries")
print(f"  فلم → {_uri_to_label.get('فلم', 'NOT FOUND')}")

# ── Normalise type labels ────────────────────────────────────────────────────
def normalise_type(t: str) -> str:
    if not t:
        return t
    if t.startswith("http"):
        t = t.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
    t = t.replace("xsd:", "")
    t = _uri_to_label.get(t, t)
    return t

LITERAL_TYPES = {"number", "Year", "Date", "Code", "Currency", "Colour",
                 "Language", "decimal", "gYear", "date", "string", "integer",
                 "float", "double", "boolean"}

def is_literal_type(t: str) -> bool:
    return normalise_type(t).lower() in {x.lower() for x in LITERAL_TYPES}

# ── Transitive IsA map ──────────────────────────────────────────────────────
def build_superclass_map(hierarchy_list: list) -> dict:
    parent_of = {}
    for h in hierarchy_list:
        if h.get("id") == "IsA":
            parent_of[h["domain"]] = h["range"]
    superclasses = {}
    for cls in set(list(parent_of.keys()) + list(parent_of.values())):
        ancestors = {cls}
        cur = cls
        while cur in parent_of:
            cur = parent_of[cur]
            ancestors.add(cur)
        superclasses[cls] = ancestors
    return superclasses

# ── Check one triple against ALL definitions of a relation ───────────────────
def check_triple_schema(relation: str, subj_type: str, obj_type: str,
                        rel_defs: dict, superclasses: dict):
    """
    rel_defs maps relation_label → list[{domain, range}].
    A triple is valid if it satisfies ANY of the definitions for the relation.
    """
    subj_type = normalise_type(subj_type)
    obj_type  = normalise_type(obj_type)

    if relation not in rel_defs:
        return dict(found=False, domain_ok=None, range_ok=None,
                    expected_domains=None, expected_ranges=None)

    defs = rel_defs[relation]
    subj_ancestors = superclasses.get(subj_type, {subj_type})

    # Check each definition — valid if ANY passes
    best_domain_ok = False
    best_range_ok = False
    any_fully_ok = False

    for d in defs:
        domain = d["domain"]
        range_ = d["range"]

        d_ok = domain in subj_ancestors
        if is_literal_type(range_):
            r_ok = is_literal_type(obj_type)
        else:
            obj_ancestors = superclasses.get(obj_type, {obj_type})
            r_ok = range_ in obj_ancestors

        if d_ok:
            best_domain_ok = True
        if r_ok:
            best_range_ok = True
        if d_ok and r_ok:
            any_fully_ok = True

    domains = sorted({d["domain"] for d in defs})
    ranges  = sorted({d["range"]  for d in defs})

    return dict(found=True,
                domain_ok=best_domain_ok,
                range_ok=best_range_ok,
                fully_ok=any_fully_ok,
                expected_domains="|".join(domains),
                expected_ranges="|".join(ranges))


# ── Main analysis ────────────────────────────────────────────────────────────
total_triples = 0
triples_with_schema = 0
relation_not_found = 0
domain_violations = 0
range_violations = 0
both_violations = 0
correct = 0

violation_details = []
per_category = defaultdict(lambda: {"total": 0, "correct": 0,
                                     "domain_viol": 0, "range_viol": 0,
                                     "not_found": 0})

for _, row in df.iterrows():
    entry_id = row["entry_id"]
    category = row["category"]
    pred_triples = row["pred_triples"]
    pred_schemas = row.get("pred_schemas", [])

    if not isinstance(pred_schemas, list):
        pred_schemas = []

    ont_path = os.path.join(ONTOLOGY_DIR, f"{category}.json")
    with open(ont_path, "r", encoding="utf-8") as f:
        ont = json.load(f)

    # Build a list of definitions per relation label (handles duplicates)
    rel_defs = defaultdict(list)
    for r in ont.get("relation", []):
        rel_defs[r["label"]].append(r)

    superclasses = build_superclass_map(ont.get("hierarchy", []))

    for t_idx, triple in enumerate(pred_triples):
        total_triples += 1
        cat_stats = per_category[category]
        cat_stats["total"] += 1

        if t_idx >= len(pred_schemas):
            continue
        schema = pred_schemas[t_idx]
        triples_with_schema += 1

        result = check_triple_schema(
            triple["relation"], schema["subject"], schema["object"],
            rel_defs, superclasses
        )

        if not result["found"]:
            relation_not_found += 1
            cat_stats["not_found"] += 1
            violation_details.append((
                entry_id, t_idx, triple["relation"],
                "relation_not_found", ""
            ))
            continue

        d_ok = result["domain_ok"]
        r_ok = result["range_ok"]

        if d_ok and r_ok:
            correct += 1
            cat_stats["correct"] += 1
        else:
            if not d_ok and not r_ok:
                both_violations += 1
            if not d_ok:
                domain_violations += 1
                cat_stats["domain_viol"] += 1
                violation_details.append((
                    entry_id, t_idx, triple["relation"],
                    "domain_mismatch",
                    f"predicted={normalise_type(schema['subject'])}, "
                    f"expected={result['expected_domains']}"
                ))
            if not r_ok:
                range_violations += 1
                cat_stats["range_viol"] += 1
                violation_details.append((
                    entry_id, t_idx, triple["relation"],
                    "range_mismatch",
                    f"predicted={normalise_type(schema['object'])}, "
                    f"expected={result['expected_ranges']}"
                ))

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"Total predicted triples         : {total_triples}")
print(f"  with schema info              : {triples_with_schema}")
print(f"  relation not in ontology      : {relation_not_found}")
print(f"  schema correct (domain+range) : {correct}")
print(f"  domain violations             : {domain_violations}")
print(f"  range  violations             : {range_violations}")
print(f"  both domain+range violated    : {both_violations}")
if triples_with_schema:
    print(f"\nViolation rate: {(domain_violations + range_violations) / triples_with_schema * 100:.1f}% "
          f"of triples with schemas  ({domain_violations + range_violations}/{triples_with_schema})")


URI→label map: 767 entries
  فلم → Film
Total predicted triples         : 1831
  with schema info              : 1831
  relation not in ontology      : 0
  schema correct (domain+range) : 1778
  domain violations             : 14
  range  violations             : 47
  both domain+range violated    : 8

Violation rate: 3.3% of triples with schemas  (61/1831)


In [16]:
# ── Per-category violation table ──────────────────────────────────────────────
cat_rows = []
for cat in sorted(per_category):
    s = per_category[cat]
    total = s["total"]
    viol = s["domain_viol"] + s["range_viol"]
    cat_rows.append({
        "Category": cat,
        "Triples": total,
        "Correct": s["correct"],
        "Domain↓": s["domain_viol"],
        "Range↓": s["range_viol"],
        "NotFound": s["not_found"],
        "Violation%": round(viol / total * 100, 1) if total else 0.0,
    })

cat_df = pd.DataFrame(cat_rows).set_index("Category")
cat_df.style.background_gradient(subset=["Violation%"], cmap="Reds")


,Triples,Correct,Domain↓,Range↓,NotFound,Violation%
Category,,,,,,
1_Airport,35,35,0,0,0,0.000000
1_Artist,32,32,0,0,0,0.000000
1_Astronaut,8,8,0,0,0,0.000000
1_Athlete,32,32,0,0,0,0.000000
1_Building,26,24,0,2,0,7.700000
1_CelestialBody,19,16,0,3,0,15.800000
1_City,27,27,0,0,0,0.000000
1_ComicsCharacter,11,11,0,0,0,0.000000
1_Company,10,10,0,0,0,0.000000


In [17]:
# ── Sample violations ─────────────────────────────────────────────────────────
viol_df = pd.DataFrame(violation_details,
                        columns=["entry_id", "triple_idx", "relation",
                                 "issue", "detail"])
print(f"Total violation records: {len(viol_df)}")
print(f"\nDomain mismatches: {(viol_df['issue']=='domain_mismatch').sum()}")
print(f"Range  mismatches: {(viol_df['issue']=='range_mismatch').sum()}")
print(f"Relation not found: {(viol_df['issue']=='relation_not_found').sum()}")

print("\n── First 20 domain/range violations ──")
viol_df[viol_df["issue"] != "relation_not_found"].head(20)


Total violation records: 61

Domain mismatches: 14
Range  mismatches: 47
Relation not found: 0

── First 20 domain/range violations ──


,entry_id,triple_idx,relation,issue,detail
0,1_Building_dev_10,0,cost,range_mismatch,"predicted=TopicalConcept, expected=number"
1,1_Building_dev_22,0,completionYear,range_mismatch,"predicted=TopicalConcept, expected=Year"
2,1_CelestialBody_dev_7,0,discoverer,range_mismatch,"predicted=Organisation, expected=Person|Place|..."
3,1_CelestialBody_dev_8,0,discoverer,range_mismatch,"predicted=Organisation, expected=Person|Place|..."
4,1_CelestialBody_dev_18,0,discoverer,range_mismatch,"predicted=Organisation, expected=Person|Place|..."
5,1_Politician_dev_3,0,commander,domain_mismatch,"predicted=MilitaryPerson, expected=Event|Organ..."
6,1_Politician_dev_3,0,commander,range_mismatch,"predicted=MilitaryConflict, expected=Person"
7,2_Airport_dev_2,0,elevationAboveTheSeaLevelInMetres,range_mismatch,"predicted=ArchitecturalStructure, expected=number"
8,2_Airport_dev_3,1,owner,domain_mismatch,"predicted=Settlement, expected=ArchitecturalSt..."
9,2_Airport_dev_3,1,owner,range_mismatch,"predicted=Airport, expected=Organisation|Place"


In [18]:
# ── Top 10 violating categories ───────────────────────────────────────────────
top10 = cat_df.nlargest(10, "Violation%")[["Triples", "Domain↓", "Range↓", "Violation%"]]
print("Top-10 categories by violation %:")
print(top10.to_string())

# ── Inspect violations for the worst category ────────────────────────────────
worst_cat = top10.index[0]
worst_viols = viol_df[viol_df["entry_id"].str.startswith(worst_cat + "_")]
print(f"\n── {worst_cat} violations ({len(worst_viols)} total) ──")
for _, v in worst_viols.head(15).iterrows():
    print(f"  {v['issue']:20s}  {v['relation']:30s}  {v['detail']}")


Top-10 categories by violation %:
                 Triples  Domain↓  Range↓  Violation%
Category                                             
2_Airport             46        1      11        26.1
2_SportsTeam          38        4       4        21.1
3_City                71        3      10        18.3
1_CelestialBody       19        0       3        15.8
3_WrittenWork         73        0       8        11.0
3_Airport             69        3       3         8.7
1_Building            26        0       2         7.7
3_MusicalWork         14        0       1         7.1
2_City                46        0       3         6.5
1_Politician          33        1       1         6.1

── 2_Airport violations (12 total) ──
  range_mismatch        elevationAboveTheSeaLevelInMetres  predicted=ArchitecturalStructure, expected=number
  domain_mismatch       owner                           predicted=Settlement, expected=ArchitecturalStructure
  range_mismatch        owner                           pred

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

FUZZY_THRESHOLD = 0.85
SYNONYM_GROUPS  = [
    {"currentTeam", "formerTeam", "club", "currentClub"},
    {"country", "isPartOf"},
]

# Load once — reused for all similarity calls
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Done.")

# ── helpers ───────────────────────────────────────────────────────────────────
def exact_match(pred: dict, gold: dict) -> bool:
    return (pred["subject"].lower()  == gold["subject"].lower()  and
            pred["relation"].lower() == gold["relation"].lower() and
            pred["object"].lower()   == gold["object"].lower())

def relations_are_synonyms(r1: str, r2: str) -> bool:
    r1, r2 = r1.lower(), r2.lower()
    return any(r1 in g and r2 in g for g in
               [{x.lower() for x in grp} for grp in SYNONYM_GROUPS])

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def fuzzy_match(pred: dict, gold: dict,
                emb_cache: dict, threshold: float = FUZZY_THRESHOLD) -> bool:
    """Relation must match exactly; subject & object matched by embedding similarity."""
    if pred["relation"].lower() != gold["relation"].lower():
        return False
    for key in ("subject", "object"):
        pv, gv = pred[key].lower(), gold[key].lower()
        if pv == gv:
            continue
        if pv not in emb_cache:
            emb_cache[pv] = embedder.encode(pv, convert_to_numpy=True)
        if gv not in emb_cache:
            emb_cache[gv] = embedder.encode(gv, convert_to_numpy=True)
        if cosine(emb_cache[pv], emb_cache[gv]) < threshold:
            return False
    return True

def synonym_match(pred: dict, gold: dict,
                  emb_cache: dict, threshold: float = FUZZY_THRESHOLD) -> bool:
    """Like fuzzy_match but also accepts synonym relations."""
    rel_ok = (pred["relation"].lower() == gold["relation"].lower() or
              relations_are_synonyms(pred["relation"], gold["relation"]))
    if not rel_ok:
        return False
    for key in ("subject", "object"):
        pv, gv = pred[key].lower(), gold[key].lower()
        if pv == gv:
            continue
        if pv not in emb_cache:
            emb_cache[pv] = embedder.encode(pv, convert_to_numpy=True)
        if gv not in emb_cache:
            emb_cache[gv] = embedder.encode(gv, convert_to_numpy=True)
        if cosine(emb_cache[pv], emb_cache[gv]) < threshold:
            return False
    return True

def count_matched_full(pred_triples: list, gold_triples: list,
                       emb_cache: dict):
    """
    Returns (exact, fuzzy_extra, synonym_extra, fuzzy_corrections, synonym_corrections).
    fuzzy_corrections  — list of (pred, gold) pairs fixed by fuzzy but not exact.
    synonym_corrections — list of (pred, gold) pairs fixed by synonym but not fuzzy.
    No double-counting: each gold triple matched at most once (best match wins).
    """
    used_exact = set(); used_fuzzy = set(); used_syn = set()
    exact_n = fuzzy_n = syn_n = 0
    fuzzy_corrections = []
    synonym_corrections = []

    # exact pass
    for p in pred_triples:
        for j, g in enumerate(gold_triples):
            if j not in used_exact and exact_match(p, g):
                exact_n += 1
                used_exact.add(j)
                break

    # fuzzy pass (only on gold not already matched exactly)
    used_fuzzy = set(used_exact)
    for p in pred_triples:
        # skip if this pred was already counted in exact
        already = any(j in used_exact and exact_match(p, gold_triples[j])
                      for j in used_exact)
        if already:
            continue
        for j, g in enumerate(gold_triples):
            if j not in used_fuzzy and fuzzy_match(p, g, emb_cache):
                fuzzy_n += 1
                used_fuzzy.add(j)
                fuzzy_corrections.append({"pred": p, "gold": g})
                break

    # synonym pass (only on gold not already matched by exact or fuzzy)
    used_syn = set(used_fuzzy)
    for p in pred_triples:
        already_exact = any(j in used_exact and exact_match(p, gold_triples[j])
                            for j in used_exact)
        already_fuzzy = any(c["pred"] == p for c in fuzzy_corrections)
        if already_exact or already_fuzzy:
            continue
        for j, g in enumerate(gold_triples):
            if j not in used_syn and synonym_match(p, g, emb_cache):
                syn_n += 1
                used_syn.add(j)
                synonym_corrections.append({"pred": p, "gold": g})
                break

    return exact_n, fuzzy_n, syn_n, fuzzy_corrections, synonym_corrections

# ── run over full dataframe ───────────────────────────────────────────────────
print("Computing matches (embedding calls may take a moment)...")
emb_cache = {}

results = df.apply(
    lambda r: count_matched_full(r["pred_triples"], r["gold_triples"], emb_cache),
    axis=1
)

df["matched_exact"]        = results.apply(lambda x: x[0])
df["matched_fuzzy_extra"]  = results.apply(lambda x: x[1])
df["matched_syn_extra"]    = results.apply(lambda x: x[2])
df["fuzzy_corrections"]    = results.apply(lambda x: x[3])
df["synonym_corrections"]  = results.apply(lambda x: x[4])

# cumulative matched columns
df["matched_fuzzy"]    = df["matched_exact"] + df["matched_fuzzy_extra"]
df["matched_synonym"]  = df["matched_fuzzy"] + df["matched_syn_extra"]

print(f"Exact matched   : {df['matched_exact'].sum()}")
print(f"+ Fuzzy extra   : {df['matched_fuzzy_extra'].sum()}  → {df['matched_fuzzy'].sum()} total")
print(f"+ Synonym extra : {df['matched_syn_extra'].sum()}  → {df['matched_synonym'].sum()} total")

Loading embedding model...
Done.
Computing matches (embedding calls may take a moment)...
Exact matched   : 1488
+ Fuzzy extra   : 41  → 1529 total
+ Synonym extra : 32  → 1561 total


In [4]:
def micro_prf(group, matched_col):
    m       = group[matched_col].sum()
    p_total = group["n_pred"].sum()
    g_total = group["n_gold"].sum()
    prec = m / p_total if p_total else 0.0
    rec  = m / g_total if g_total else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return pd.Series({"Precision": prec, "Recall": rec, "F1": f1,
                      "Matched": int(m), "Gold": int(g_total),
                      "Predicted": int(p_total), "Entries": len(group)})

cat_exact   = df.groupby("category").apply(micro_prf, matched_col="matched_exact",   include_groups=False)
cat_fuzzy   = df.groupby("category").apply(micro_prf, matched_col="matched_fuzzy",   include_groups=False)
cat_synonym = df.groupby("category").apply(micro_prf, matched_col="matched_synonym", include_groups=False)

# side-by-side F1
comparison = pd.DataFrame({
    "F1_exact":   cat_exact["F1"],
    "F1_fuzzy":   cat_fuzzy["F1"],
    "F1_synonym": cat_synonym["F1"],
    "Δ_fuzzy":    cat_fuzzy["F1"]   - cat_exact["F1"],
    "Δ_synonym":  cat_synonym["F1"] - cat_fuzzy["F1"],
    "Gold":       cat_exact["Gold"],
    "Entries":    cat_exact["Entries"],
}).sort_index()

comparison.style.format({c: "{:.3f}" for c in comparison.columns if c not in ("Gold","Entries")})

,F1_exact,F1_fuzzy,F1_synonym,Δ_fuzzy,Δ_synonym,Gold,Entries
category,,,,,,,
1_Airport,0.943,0.943,0.943,0.000,0.000,35.000000,35.000000
1_Artist,0.969,0.969,0.969,0.000,0.000,32.000000,32.000000
1_Astronaut,0.750,0.750,0.750,0.000,0.000,8.000000,8.000000
1_Athlete,0.688,0.906,0.906,0.219,0.000,32.000000,32.000000
1_Building,0.784,0.784,0.863,0.000,0.078,25.000000,25.000000
1_CelestialBody,0.947,0.947,0.947,0.000,0.000,19.000000,19.000000
1_City,0.815,0.815,0.852,0.000,0.037,27.000000,27.000000
1_ComicsCharacter,1.000,1.000,1.000,0.000,0.000,11.000000,11.000000
1_Company,0.600,0.800,0.800,0.200,0.000,10.000000,10.000000


In [5]:
def overall_prf(matched_col):
    return micro_prf(df, matched_col)

ov_exact   = overall_prf("matched_exact")
ov_fuzzy   = overall_prf("matched_fuzzy")
ov_synonym = overall_prf("matched_synonym")

for label, ov in [("EXACT", ov_exact), ("FUZZY (≥0.9 sim)", ov_fuzzy), ("SYNONYM-AWARE", ov_synonym)]:
    print(f"\n{'─'*50}")
    print(f"{label}")
    print(f"  Precision : {ov['Precision']:.4f}")
    print(f"  Recall    : {ov['Recall']:.4f}")
    print(f"  F1        : {ov['F1']:.4f}")
    print(f"  Matched   : {int(ov['Matched'])} / {int(ov['Gold'])} gold, {int(ov['Predicted'])} pred")

print(f"\n  Fuzzy  fixed {int(ov_fuzzy['Matched']  - ov_exact['Matched'])} extra triples "
      f"(F1 +{ov_fuzzy['F1']-ov_exact['F1']:.4f})")
print(f"  Synonym fixed {int(ov_synonym['Matched'] - ov_fuzzy['Matched'])} extra triples "
      f"(F1 +{ov_synonym['F1']-ov_fuzzy['F1']:.4f})")


──────────────────────────────────────────────────
EXACT
  Precision : 0.8127
  Recall    : 0.7754
  F1        : 0.7936
  Matched   : 1488 / 1919 gold, 1831 pred

──────────────────────────────────────────────────
FUZZY (≥0.9 sim)
  Precision : 0.8351
  Recall    : 0.7968
  F1        : 0.8155
  Matched   : 1529 / 1919 gold, 1831 pred

──────────────────────────────────────────────────
SYNONYM-AWARE
  Precision : 0.8525
  Recall    : 0.8134
  F1        : 0.8325
  Matched   : 1561 / 1919 gold, 1831 pred

  Fuzzy  fixed 41 extra triples (F1 +0.0219)
  Synonym fixed 32 extra triples (F1 +0.0171)


In [6]:
fuzzy_rows = df[df["matched_fuzzy_extra"] > 0][["entry_id","category","input_text","fuzzy_corrections"]]

for _, row in fuzzy_rows.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['entry_id']}  [{row['category']}]")
    print(f"Text: {row['input_text'][:120]}")
    for c in row["fuzzy_corrections"]:
        print(f"  PRED : ({c['pred']['subject']}, {c['pred']['relation']}, {c['pred']['object']})")
        print(f"  GOLD : ({c['gold']['subject']}, {c['gold']['relation']}, {c['gold']['object']})")
        print()


1_Athlete_dev_5  [1_Athlete]
Text: Alaa Abdul-Zahra played for the club Al-Zawra'a SC.
  PRED : (Alaa_Abdul_Zahra, club, Al_Zawraa_SC)
  GOLD : (Alaa_Abdul_Zahra, club, Al_Zawra_a_SC)


1_Athlete_dev_13  [1_Athlete]
Text: The Chairman of A.C. Milan is Silvio Berlusconi.
  PRED : (AC_Milan, chairman, Silvio_Berlusconi)
  GOLD : (A_C_Milan, chairman, Silvio_Berlusconi)


1_Athlete_dev_19  [1_Athlete]
Text: The United Petrotrin F.C. is playing in Palo Seco Velodrome.
  PRED : (United_Petrotrin_FC, ground, Palo_Seco_Velodrome)
  GOLD : (United_Petrotrin_F_C, ground, Palo_Seco_Velodrome)


1_Athlete_dev_20  [1_Athlete]
Text: Alessio Romagnoli plays for A.C. Milan.
  PRED : (Alessio_Romagnoli, club, AC_Milan)
  GOLD : (Alessio_Romagnoli, club, A_C_Milan)


1_Athlete_dev_21  [1_Athlete]
Text: Alan Martin played football for Accrington Stanley F.C.
  PRED : (Alan_Martin, club, Accrington_Stanley_FC)
  GOLD : (Alan_Martin, club, Accrington_Stanley_F_C)


1_Athlete_dev_23  [1_Athlete]
Text: Pal

In [7]:
syn_rows = df[df["matched_syn_extra"] > 0][["entry_id","category","input_text","synonym_corrections"]]

for _, row in syn_rows.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['entry_id']}  [{row['category']}]")
    print(f"Text: {row['input_text'][:120]}")
    for c in row["synonym_corrections"]:
        print(f"  PRED : ({c['pred']['subject']}, {c['pred']['relation']}, {c['pred']['object']})")
        print(f"  GOLD : ({c['gold']['subject']}, {c['gold']['relation']}, {c['gold']['object']})")
        print()


1_Building_dev_1  [1_Building]
Text: Ahmedabad is in India.
  PRED : (Ahmedabad, isPartOf, India)
  GOLD : (Ahmedabad, country, India)


1_Building_dev_3  [1_Building]
Text: Dublin is in the Republic of Ireland.
  PRED : (Dublin, isPartOf, Republic_of_Ireland)
  GOLD : (Dublin, country, Republic_of_Ireland)


1_City_dev_19  [1_City]
Text: Texas is in the United States.
  PRED : (Texas, isPartOf, United_States)
  GOLD : (Texas, country, United_States)


2_Airport_dev_19  [2_Airport]
Text: Al-Taqaddum Air Base serves the city of Fallujah which is in Iraq.
  PRED : (Fallujah, isPartOf, Iraq)
  GOLD : (Fallujah, country, Iraq)


2_Athlete_dev_4  [2_Athlete]
Text: Alessio Romagnoli played for the Italy national under-16 football team coached by Daniele Zoratto.
  PRED : (Alessio_Romagnoli, formerTeam, Italy_national_under_16_football_team)
  GOLD : (Alessio_Romagnoli, club, Italy_national_under_16_football_team)


2_Athlete_dev_14  [2_Athlete]
Text: Former clubs of Alessio Romagnoli includ

In [8]:
# uses matched_synonym as the final "correct" count
errors_inds = df[df["matched_synonym"] != df["n_gold"]].index.tolist()

def index_generator():
    for ind in errors_inds:
        yield ind

gen = index_generator()

In [ ]:
import json as _json, os as _os, xml.etree.ElementTree as _ET

idx = next(gen)
row = df.iloc[idx]

# ── Load gold schema from ontology JSON ──────────────────────────────────────
_schema_path = _os.path.join("OSKGC", "ontologies", "json", f"{row['category']}.json")
_gold_schema = _json.load(open(_schema_path))
_gold_rel_map = {r["label"]: r for r in _gold_schema.get("relation", [])}
_used_relations = list(dict.fromkeys(t["relation"] for t in row["gold_triples"]))

# ── Load per-entry schemas from OSKGC/data/{split}/{category}.xml ────────────
# entry_id format: {category}_{split}_{n}  →  split is second-to-last segment
_parts = row["entry_id"].rsplit("_", 2)   # e.g. ["1_Airport", "dev", "1"]
_split = _parts[1] if len(_parts) == 3 else "dev"
_xml_path = _os.path.join("OSKGC", "data", _split, f"{row['category']}.xml")
_entry_schemas = []
if _os.path.exists(_xml_path):
    _tree = _ET.parse(_xml_path)
    for _entry in _tree.findall(".//entry"):
        if _entry.get("id") == row["entry_id"]:
            for _s in _entry.findall(".//schema"):
                _entry_schemas.append({
                    "subject": _s.findtext("sub", ""),
                    "relation": _s.findtext("rel", ""),
                    "object":  _s.findtext("obj", ""),
                })
            break

print("=" * 80)
print(f"Index: {idx} | Entry ID: {row['entry_id']}")
print(f"Category: {row['category']} | "
      f"Exact: {row['matched_exact']}/{row['n_gold']} | "
      f"Fuzzy: {row['matched_fuzzy']}/{row['n_gold']} | "
      f"Synonym: {row['matched_synonym']}/{row['n_gold']}")
print("-" * 80)
print(f"Input: {row['input_text']}")
print("-" * 80)
print("GOLD TRIPLES:")
for i, t in enumerate(row["gold_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("-" * 80)
print("PRED TRIPLES:")
for i, t in enumerate(row["pred_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("-" * 80)
print("PRED SCHEMA (predicted subject/object types):")
for i, s in enumerate(row["pred_schemas"], 1):
    print(f"  {i}. subject={s['subject']}  object={s['object']}")
print("-" * 80)
print("GOLD SCHEMA — per-entry (from data XML):")
if _entry_schemas:
    for i, s in enumerate(_entry_schemas, 1):
        print(f"  {i}. [{s['relation']}]  subject={s['subject']}  object={s['object']}")
else:
    print("  (not found)")
print("-" * 80)
print("GOLD SCHEMA — ontology (domain/range for gold relations):")
for i, rel in enumerate(_used_relations, 1):
    r = _gold_rel_map.get(rel)
    if r:
        print(f"  {i}. [{rel}]  domain={r['domain']}  range={r['range']}")
    else:
        print(f"  {i}. [{rel}]  (not found in ontology)")
print("=" * 80)


Index: 74 | Entry ID: 1_Astronaut_dev_8
Category: 1_Astronaut | Exact: 0/1 | Fuzzy: 0/1 | Synonym: 0/1
--------------------------------------------------------------------------------
Input: William Anders served as a crew member on Apollo 8.
--------------------------------------------------------------------------------
GOLD TRIPLES:
  1. (William_Anders, mission, Apollo_8)
--------------------------------------------------------------------------------
PRED TRIPLES:
  1. (Apollo_8, crewMembers, William_Anders)


In [125]:
def calculate_embedding_similarity(text1: str, text2: str) -> float:
    """Calculate cosine similarity between embeddings of two texts."""
    emb1 = embedder.encode(text1, convert_to_numpy=True)
    emb2 = embedder.encode(text2, convert_to_numpy=True)
    return cosine(emb1, emb2)

similarity = calculate_embedding_similarity("Tudor_Revival", "Tudor_Revival_architecture")
print(f"Similarity: {similarity:.4f}")

Similarity: 0.7963
